# NYC 311 / collisions — S3 Parquet spot check

Read one `year=…/month=…` partition from S3 and inspect column names and sample rows.

In [3]:
from pathlib import Path
import os

import pandas as pd
import pyarrow.parquet as pq
from pyarrow.fs import S3FileSystem

# --- edit these ---
DATASET = "nyc_311"   # nyc_311 | nypd_collisions
YEAR = 2024
MONTH = 1             # 1-12

# Load .env (same keys as load_nyc_data.py)
_env = Path(".env")
if _env.is_file():
    for line in _env.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k, v = k.strip(), v.strip()
        if k and k not in os.environ:
            os.environ[k] = v

BUCKET = os.environ.get("NYC_DATA_BUCKET", "gtp-nyc-data-bucket").strip()
PREFIX = os.environ.get("NYC_DATA_PREFIX", "").strip().rstrip("/")
prefix_part = f"{PREFIX}/" if PREFIX else ""
S3_KEY = f"{prefix_part}{DATASET}/year={YEAR}/month={MONTH:02d}/part-0.parquet"
S3_URI = f"s3://{BUCKET}/{S3_KEY}"

print(S3_URI)

s3://gtp-nyc-data-bucket/nyc_311/year=2024/month=01/part-0.parquet


In [5]:
pd.set_option('display.max_columns', 500)

In [6]:
fs = S3FileSystem()
table = pq.read_table(f"{BUCKET}/{S3_KEY}", filesystem=fs)
df = table.to_pandas()

print(f"rows: {len(df):,}  cols: {len(df.columns)}")
print("\n--- column names ---")
for c in df.columns:
    print(c)

print("\n--- first 5 rows (all columns) ---")
display(df.head())

rows: 287,186  cols: 48

--- column names ---
address_type
agency
agency_name
bbl
borough
bridge_highway_direction
bridge_highway_name
bridge_highway_segment
city
closed_date
community_board
complaint_type
council_district
created_date
cross_street_1
cross_street_2
descriptor
descriptor_2
due_date
facility_type
h3_r10
h3_r8
h3_r9
incident_address
incident_zip
intersection_street_1
intersection_street_2
landmark
latitude
location_type
longitude
open_data_channel_type
park_borough
park_facility_name
police_precinct
resolution_action_updated_date
resolution_description
road_ramp
status
street_name
taxi_company_borough
taxi_pick_up_location
unique_key
vehicle_type
x_coordinate_state_plane
y_coordinate_state_plane
year
month

--- first 5 rows (all columns) ---


,address_type,agency,agency_name,bbl,borough,bridge_highway_direction,bridge_highway_name,bridge_highway_segment,city,closed_date,community_board,complaint_type,council_district,created_date,cross_street_1,cross_street_2,descriptor,descriptor_2,due_date,facility_type,h3_r10,h3_r8,h3_r9,incident_address,incident_zip,intersection_street_1,intersection_street_2,landmark,latitude,location_type,longitude,open_data_channel_type,park_borough,park_facility_name,police_precinct,resolution_action_updated_date,resolution_description,road_ramp,status,street_name,taxi_company_borough,taxi_pick_up_location,unique_key,vehicle_type,x_coordinate_state_plane,y_coordinate_state_plane,year,month
0,ADDRESS,NYPD,New York City Police Department,2032140044,BRONX,NaN,NaN,NaN,BRONX,2024-01-01T01:00:13.000,07 BRONX,Illegal Parking,14,2024-01-01T00:04:14.000,WEST 190 STREET,WEST 192 STREET,Blocked Hydrant,NaN,NaN,NaN,8a2a100acd2ffff,882a100acdfffff,892a100acd3ffff,2535 GRAND AVENUE,10468,WEST 190 STREET,WEST 192 STREET,GRAND AVENUE,40.865310686039486,Street/Sidewalk,-73.90138225011205,MOBILE,BRONX,Unspecified,Precinct 52,2024-01-01T01:00:15.000,The Police Department issued a summons in resp...,NaN,Closed,GRAND AVENUE,NaN,NaN,59886871,NaN,1011527,254549,2024,1
1,ADDRESS,NYPD,New York City Police Department,4014927501,QUEENS,NaN,NaN,NaN,ELMHURST,2024-01-01T01:27:12.000,04 QUEENS,Illegal Parking,25,2024-01-01T00:59:23.000,81 STREET,BAXTER AVENUE,Posted Parking Sign Violation,NaN,NaN,NaN,8a2a100c6b27fff,882a100c45fffff,892a100c6b3ffff,81-09 41 AVENUE,11373,81 STREET,BAXTER AVENUE,41 AVENUE,40.74591558443856,Street/Sidewalk,-73.88419276835766,PHONE,QUEENS,Unspecified,Precinct 110,2024-01-01T01:27:15.000,The Police Department responded to the complai...,NaN,Closed,41 AVENUE,NaN,NaN,59886906,NaN,1016339,211055,2024,1
2,ADDRESS,NYPD,New York City Police Department,3072730025,BROOKLYN,NaN,NaN,NaN,BROOKLYN,2024-01-01T01:09:19.000,13 BROOKLYN,Illegal Parking,48,2024-01-01T00:52:54.000,NEPTUNE AVENUE,WEST BRIGHTON AVENUE,Posted Parking Sign Violation,NaN,NaN,NaN,8a2a10741547fff,882a107415fffff,892a1074157ffff,2928 MARSHA RAPAPORT WAY,11224,NEPTUNE AVENUE,WEST BRIGHTON AVENUE,WEST 5 STREET,40.57822610767071,Street/Sidewalk,-73.9723457475236,MOBILE,BROOKLYN,Unspecified,Precinct 60,2024-01-01T01:09:23.000,The Police Department responded to the complai...,NaN,Closed,MARSHA RAPAPORT WAY,NaN,NaN,59886911,NaN,991932,149941,2024,1
3,ADDRESS,NYPD,New York City Police Department,4119740010,QUEENS,NaN,NaN,NaN,JAMAICA,2024-01-01T01:03:11.000,12 QUEENS,Blocked Driveway,28,2024-01-01T00:44:31.000,144 STREET,145 STREET,No Access,NaN,NaN,NaN,8a2a100e960ffff,882a100e97fffff,892a100e963ffff,144-28 LINDEN BOULEVARD,11436,144 STREET,145 STREET,LINDEN BOULEVARD,40.684755327662195,Street/Sidewalk,-73.79931420196795,PHONE,QUEENS,Unspecified,Precinct 113,2024-01-01T01:03:14.000,The Police Department responded to the complai...,NaN,Closed,LINDEN BOULEVARD,NaN,NaN,59887007,NaN,1039909,188815,2024,1
4,ADDRESS,NYPD,New York City Police Department,3045860945,BROOKLYN,NaN,NaN,NaN,BROOKLYN,2024-01-01T01:17:24.000,05 BROOKLYN,Blocked Driveway,42,2024-01-01T00:41:16.000,ESSEX STREET,BERRIMAN STREET,Partial Access,NaN,NaN,NaN,8a2a100cb857fff,882a100cb9fffff,892a100cb87ffff,598 SCHROEDERS AVENUE,11239,ESSEX STREET,BERRIMAN STREET,SCHROEDERS AVENUE,40.65599325559804,Street/Sidewalk,-73.87040829977123,ONLINE,BROOKLYN,Unspecified,Precinct 75,2024-01-01T01:17:29.000,The Police Department responded to the complai...,NaN,Closed,SCHROEDERS AVENUE,NaN,NaN,59887011,NaN,1020207,178299,2024,1


In [7]:
# H3 columns (should be present after a reload with current load_nyc_data.py)
h3_cols = [c for c in df.columns if c.startswith("h3_r")]
print("H3 columns:", h3_cols)
if h3_cols:
    display(df[h3_cols].head(10))
else:
    print("No h3_r* columns — partition may be from an older load.")

H3 columns: ['h3_r10', 'h3_r8', 'h3_r9']


,h3_r10,h3_r8,h3_r9
0,8a2a100acd2ffff,882a100acdfffff,892a100acd3ffff
1,8a2a100c6b27fff,882a100c45fffff,892a100c6b3ffff
2,8a2a10741547fff,882a107415fffff,892a1074157ffff
3,8a2a100e960ffff,882a100e97fffff,892a100e963ffff
4,8a2a100cb857fff,882a100cb9fffff,892a100cb87ffff
5,8a2a100dec9ffff,882a100dedfffff,892a100de53ffff
6,8a2a107548affff,882a107549fffff,892a107548bffff
7,8a2a100c4d87fff,882a100c4dfffff,892a100c4dbffff
8,8a2a1077362ffff,882a107737fffff,892a1077363ffff
9,8a2a106202cffff,882a106203fffff,892a106202fffff
